# 🧠 Neural Networks & Deep Learning
**ISLP Chapter 10 · Data Pattern Recognition for the Rest of Us**

> Neural networks learn hierarchical representations from data. For tabular business data, even a shallow network often outperforms linear models. For images, text, and sequences, deep networks are transformative.

### Dataset
**Employee Compensation Analytics**
Predict annual salary from job title, department, years of experience, education, and performance metrics. A real HR/compensation analytics problem where neural networks can capture nonlinear interactions (e.g., seniority × department effects).

### Time: ~65 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
print("\u2713 Setup complete")

In [ ]:
# Employee Compensation Dataset
# Realistic salary data with nonlinear interactions between features
np.random.seed(42)
n = 3000

departments = ['Engineering','Finance','Marketing','Sales','Operations','HR','Legal']
job_levels  = ['Junior','Mid','Senior','Lead','Director','VP']
education   = ['High School','Associate','Bachelor','Master','PhD']

dept = np.random.choice(departments, n, p=[0.30,0.15,0.12,0.18,0.12,0.08,0.05])
level= np.random.choice(job_levels, n, p=[0.20,0.28,0.25,0.15,0.08,0.04])
edu  = np.random.choice(education, n, p=[0.05,0.08,0.45,0.32,0.10])
yoe  = np.random.randint(0, 25, n).astype(float)
perf = np.random.randint(1, 6, n).astype(float)  # 1-5 performance rating
age  = (22 + yoe + np.random.normal(0, 2, n)).clip(22, 65)

# Encode levels
dept_pay  = {'Engineering':1.3,'Finance':1.2,'Legal':1.25,'Marketing':1.0,
             'Sales':0.95,'Operations':0.9,'HR':0.88}
level_mult= {'Junior':0.6,'Mid':0.85,'Senior':1.1,'Lead':1.35,'Director':1.7,'VP':2.2}
edu_bonus = {'High School':0,'Associate':0.03,'Bachelor':0.08,'Master':0.15,'PhD':0.22}

# Nonlinear salary formula: interaction between dept, level, experience
base = 55000
salary = np.array([
    base
    * dept_pay[dept[i]]
    * level_mult[level[i]]
    * (1 + edu_bonus[edu[i]])
    * (1 + 0.025 * min(yoe[i], 15))   # experience premium plateaus
    * (1 + 0.04 * (perf[i] - 3))      # performance premium
    * np.random.lognormal(0, 0.08)     # noise
    for i in range(n)
])

df = pd.DataFrame({
    'Department': dept, 'JobLevel': level, 'Education': edu,
    'YearsExperience': yoe, 'PerformanceRating': perf,
    'Age': age, 'Salary': salary.round(-2)
})

print(f"Employee Compensation dataset: {df.shape}")
print(f"\nSalary range: ${df['Salary'].min():,.0f} — ${df['Salary'].max():,.0f}")
print(f"Median salary: ${df['Salary'].median():,.0f}")
print(f"\nBreakdown by level:")
print(df.groupby('JobLevel')['Salary'].agg(['mean','count']).sort_values('mean').round(0).to_string())

## 📐 Part 1 — From Neuron to Network

**A single neuron computes:**
```
a = f(w₀ + w₁x₁ + w₂x₂ + ... + wₚxₚ)
```
where f is a nonlinear activation function (ReLU: max(0,x)).

**Why layers matter:** Each layer transforms the data into a new representation. Layer 1 might learn "seniority × department" interactions. Layer 2 might learn how those interact with education. The network automatically discovers these combinations from data.

**For tabular data:** Even a shallow network (1-2 hidden layers) often beats linear models by capturing interactions and nonlinearities without explicit feature engineering.

In [ ]:
# Prepare features
le = LabelEncoder()
df_enc = df.copy()
for col in ['Department','JobLevel','Education']:
    df_enc[col] = le.fit_transform(df[col])

X = df_enc.drop('Salary', axis=1).values
y = np.log(df_enc['Salary'].values)  # predict log(salary) — stabilizes variance

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

# Compare: Linear Regression vs Neural Network
lr = LinearRegression().fit(X_tr_s, y_tr)
lr_pred = lr.predict(X_te_s)

nn_shallow = MLPRegressor(hidden_layer_sizes=(64,), activation='relu',
                           max_iter=500, random_state=42, early_stopping=True)
nn_shallow.fit(X_tr_s, y_tr)
nn_s_pred = nn_shallow.predict(X_te_s)

nn_deep = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu',
                        max_iter=500, random_state=42, early_stopping=True)
nn_deep.fit(X_tr_s, y_tr)
nn_d_pred = nn_deep.predict(X_te_s)

print(f"{'Model':<30} {'R²':>8} {'MAE ($)':>12} {'RMSE ($)':>12}")
print("-" * 65)
for name, pred in [('Linear Regression', lr_pred),
                   ('Shallow NN (64)', nn_s_pred),
                   ('Deep NN (128-64-32)', nn_d_pred)]:
    r2   = r2_score(y_te, pred)
    mae  = mean_absolute_error(np.exp(y_te), np.exp(pred))
    rmse = np.sqrt(mean_squared_error(np.exp(y_te), np.exp(pred)))
    print(f"  {name:<28} {r2:>8.4f} {mae:>12,.0f} {rmse:>12,.0f}")
print("\nMAE/RMSE are in original $ scale (we exponentiate log predictions)")

In [ ]:
# Visualize: actual vs predicted salary + learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs predicted
y_pred_dollars = np.exp(nn_deep.predict(X_te_s))
y_actual_dollars = np.exp(y_te)

axes[0].scatter(y_actual_dollars/1000, y_pred_dollars/1000,
                alpha=0.3, s=12, color='#1e5fa8')
max_val = max(y_actual_dollars.max(), y_pred_dollars.max()) / 1000
axes[0].plot([0,max_val],[0,max_val],'k--',lw=1.5,alpha=0.5,label='Perfect prediction')
axes[0].set_xlabel('Actual Salary ($000s)')
axes[0].set_ylabel('Predicted Salary ($000s)')
axes[0].set_title(f'Neural Network: Actual vs Predicted Salary\n'
                  f'R² = {r2_score(y_te, nn_deep.predict(X_te_s)):.3f}')
axes[0].legend()

# Residuals by department
pred_all = np.exp(nn_deep.predict(scaler.transform(X)))
residuals_pct = (pred_all - df['Salary'].values) / df['Salary'].values * 100

for dept_name in departments:
    mask = df['Department'] == dept_name
    axes[1].boxplot(residuals_pct[mask], positions=[departments.index(dept_name)],
                    widths=0.6, patch_artist=True,
                    boxprops=dict(facecolor='#1e5fa8', alpha=0.6))

axes[1].axhline(0, color='#e85d20', lw=2, ls='--')
axes[1].set_xticks(range(len(departments)))
axes[1].set_xticklabels([d[:4] for d in departments], rotation=45)
axes[1].set_ylabel('Prediction Error (%)')
axes[1].set_title('Residuals by Department\n(should be centered at 0 for unbiased model)')

plt.tight_layout(); plt.show()

In [ ]:
# Activation functions — why nonlinearity matters
x_range = np.linspace(-4, 4, 200)
activations = {
    'ReLU': np.maximum(0, x_range),
    'Sigmoid': 1/(1+np.exp(-x_range)),
    'Tanh': np.tanh(x_range),
    'Linear (no activation)': x_range,
}
fig, ax = plt.subplots(figsize=(9, 4))
colors_act = ['#1e5fa8','#e85d20','#1a7a45','#888']
for (name, vals), color in zip(activations.items(), colors_act):
    lw = 2.5 if name == 'ReLU' else 1.5
    ls = '-' if name != 'Linear (no activation)' else '--'
    ax.plot(x_range, vals, label=name, color=color, lw=lw, ls=ls)
ax.axhline(0, color='black', lw=0.5, alpha=0.3)
ax.axvline(0, color='black', lw=0.5, alpha=0.3)
ax.set_xlabel('Input value')
ax.set_ylabel('Output value')
ax.set_title('Activation Functions — why nonlinearity is essential\n'
             'Without it, a 10-layer network reduces to a single linear transformation')
ax.legend()
ax.set_ylim(-1.5, 4)
plt.tight_layout(); plt.show()
print("\n\u2714 ReLU is the default for tabular data: fast, avoids vanishing gradients")
print("   Without nonlinear activations, deep networks are mathematically equivalent")
print("   to a single linear regression — no benefit from the extra layers")

In [ ]:
answers = {
    "q1": "",  # What does a single neuron compute? Describe the two steps.
    "q2": "",  # Why do neural networks need nonlinear activation functions?
    "q3": "",  # What is backpropagation and what problem does it solve?
    "q4": "",  # On the compensation dataset, the NN outperformed linear regression. Why?
    "q5": "",  # Name one advantage and one disadvantage of neural networks vs Random Forests for tabular HR data.
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("\u274c Still empty: "+str(missing) if missing else "\u2705 Done! File \u2192 Save a copy in GitHub")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Neural Networks & Deep Learning"
# @title 🤖 AI-Graded Submission — fill in the box and click ▶ Run
# @markdown ---
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2 (one-time):** Get a free AI grading key
# @markdown - Go to [aistudio.google.com](https://aistudio.google.com) — use your **personal Gmail** (not university email — many universities block AI Studio)
# @markdown - Click **Get API key → Create API key** → copy it
# @markdown - In Colab: click the **🔑 key icon** in the left sidebar → Add secret → Name: `GEMINI_API_KEY` → paste key → toggle ON
# @markdown - Done — this persists across all 30 notebooks automatically
# @markdown
# @markdown **Step 3:** Click ▶ Run on this cell for instant AI feedback

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────────────────────
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Try to explain your reasoning in 1-2 sentences."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback on your answers."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with good detail. "
                             "Add a free Gemini key for AI analysis of your specific reasoning."),
                "tip": "Get a free key at aistudio.google.com using your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             f"Complete the remaining {n_total - n_answered} questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"\n  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    print(f"  Student  : @{username}" if username else
          "  Student  : \u26a0\ufe0f  Enter your GitHub username in the box above")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...\n")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)\n")
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 choose your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*